# Visual Classifier — Two-Stage Fine-Tuning

This notebook performs **sequential two-stage fine-tuning** of the
`dima806/ai_vs_human_generated_image_detection` ViT model:

| Stage | Dataset | Purpose |
|-------|---------|--------|
| **1** | `nebula/GenImage-arrow` (small streamed subset) | Broaden the model on diverse AI generators |
| **2** | `julienlucas/midjourney-dalle-sd-nanobananapro-dataset` | Specialise on Midjourney / DALL·E / SD |

The task is **binary classification**: `0 = Real`, `1 = AI-Generated`.

### Running on Colab from VS Code
1. Open this notebook in VS Code.
2. Select the **Google Colab** kernel (requires the Colab extension).
3. The setup cell below will clone the repo into the Colab runtime.
4. All outputs are saved inside `outputs/` in the repo folder.

### Getting outputs back to your laptop
- **Option A:** `git add`, `commit`, `push` the adapter/delta files (if small enough).
- **Option B:** Zip the `outputs/` folder and download it via Colab's file browser.
- **Option C:** Mount Google Drive and copy outputs there.

> ⚠️ **Large model checkpoints are .gitignored.** Only lightweight deltas or adapters should be committed.

---
## 0 · Configuration
Edit the values below to control dataset sizes, training hyper-parameters, etc.

In [1]:
# ── Dataset sizes for GenImage streaming subset ──
TRAIN_PER_CLASS = 1000    # 1 000 real + 1 000 AI for training
VAL_PER_CLASS   = 250     # 250 + 250 for validation
TEST_PER_CLASS  = 500     # 500 + 500 for test
SEED            = 42

# ── Model ──
BASE_MODEL = "dima806/ai_vs_human_generated_image_detection"

# ── Training hyper-parameters ──
EPOCHS_STAGE1   = 3
EPOCHS_STAGE2   = 3
BATCH_SIZE      = 8       # Colab-friendly; increase if you have more VRAM
LEARNING_RATE   = 2e-5
REPLAY_RATIO    = 0.25    # fraction of Stage-1 data mixed into Stage 2

# ── Output paths ──
STAGE1_MODEL_DIR = "outputs/models/genimage_stage1_finetuned"
STAGE2_MODEL_DIR = "outputs/models/genimage_then_julienlucas_stage2_finetuned"
EVAL_OUTPUT_DIR  = "outputs"

---
## 1 · Environment Setup (Colab / VS Code)

In [2]:
import os, sys, subprocess

# ── Detect Colab ──
IN_COLAB = "google.colab" in sys.modules
print(f"Running in Colab: {IN_COLAB}")

if IN_COLAB:
    # Install any missing packages
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
        "transformers", "datasets", "accelerate", "scikit-learn",
        "huggingface_hub", "matplotlib"])

    # Clone the repo if not already present
    REPO_DIR = "/content/Seeing-Through-Deepfakes"
    if not os.path.isdir(REPO_DIR):
        print("Cloning repository…")
        subprocess.check_call(["git", "-C",
            "https://github.com/DavidBScerri/Seeing-Through-Deepfakes.git",
            REPO_DIR], "pull")
    os.chdir(os.path.join(REPO_DIR, "src", "visual_module"))
    print(f"Working directory: {os.getcwd()}")

# ── GPU check ──
import torch
if torch.cuda.is_available():
    print(f"✅ CUDA available – {torch.cuda.get_device_name(0)}")
    USE_FP16 = True
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("✅ MPS (Apple Silicon) available")
    USE_FP16 = False   # fp16 not fully supported on MPS
else:
    print("⚠️  No GPU – training will be slow")
    USE_FP16 = False

print(f"Mixed precision (fp16): {USE_FP16}")

Running in Colab: True
Working directory: /content/Seeing-Through-Deepfakes/src/visual_module
✅ CUDA available – Tesla T4
Mixed precision (fp16): True


---
## 2 · Authentication

GenImage-arrow may require a Hugging Face token.

**Do NOT hardcode your token.** Use one of these safe methods:
- Set the `HF_TOKEN` environment variable before launching the notebook.
- Or run `huggingface_hub.login()` / `notebook_login()` interactively.
- On Colab you can also use Colab Secrets.

In [3]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN is None:
    try:
        from huggingface_hub import notebook_login
        notebook_login()          # interactive widget / prompt
        HF_TOKEN = True           # login() stores the token globally
    except Exception:
        print("⚠️  No HF token found. Public datasets will still work,"
              " but gated datasets may fail.")
        HF_TOKEN = None
else:
    print("✅ HF_TOKEN loaded from environment.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


---
## 3 · Imports & Suppress Warnings

In [4]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, AutoModelForImageClassification

# Our helper module (lives next to this notebook)
from two_stage_finetuning import (
    get_device,
    build_genimage_subset,
    load_julienlucas_dataset,
    run_training_stage,
    evaluate_model,
)
# Weight-delta helpers from the existing visual_classifier module
from visual_classifier import save_weight_delta

DEVICE = get_device()
print(f"Device: {DEVICE}")

Device: cuda


---
## 4 · Load the Base Model & Processor

In [5]:
print(f"Loading base model: {BASE_MODEL}")
processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
model     = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL, ignore_mismatched_sizes=True
).to(DEVICE)
print(f"✅ Model loaded  –  {sum(p.numel() for p in model.parameters()):,} parameters")

Loading base model: dima806/ai_vs_human_generated_image_detection


The image processor of type `ViTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Model loaded  –  85,800,194 parameters


---
## 5 · Stage 1 – GenImage Subset

### 5a · Build a balanced subset via streaming

GenImage-arrow is **very large**. We stream it and collect only the
samples we need, so nothing is downloaded in full.

In [6]:
genimage_ds = build_genimage_subset(
    train_per_class=TRAIN_PER_CLASS,
    val_per_class=VAL_PER_CLASS,
    test_per_class=TEST_PER_CLASS,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
genimage_ds

📡  Streaming GenImage-arrow (this may take a few minutes)…


Resolving data files:   0%|          | 0/1216 [00:00<?, ?it/s]

   Columns: ['image_path', 'md5', 'width', 'height', 'image']
   Example image_path: ADM/train/nature/n02966687_11178.JPEG


Resolving data files:   0%|          | 0/1216 [00:00<?, ?it/s]

   collected 500/3500…
   collected 1000/3500…
   collected 1500/3500…
   collected 2000/3500…
   collected 2500/3500…
   collected 3000/3500…
   collected 3500/3500…
   train: 2000 samples (real=1000, ai=1000)
   validation: 500 samples (real=250, ai=250)
   test: 1000 samples (real=500, ai=500)


DatasetDict({
    train: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 500
    })
    test: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 1000
    })
})

### 5b · Inspect a few samples
Verify the label mapping is correct.

In [7]:
# Show 5 examples so we can confirm the label extraction logic
for i in range(min(5, len(genimage_ds["train"]))):
    ex = genimage_ds["train"][i]
    tag = "REAL" if ex["label"] == 0 else "AI"
    print(f"  [{tag}] label={ex['label']}  path={ex.get('image_path','N/A')[:80]}")

  [AI] label=1  path=Glide/train/ai/GLIDE_1000_200_08_883_glide_00141.png
  [REAL] label=0  path=stable_diffusion_v_1_4/train/nature/n02093647_7075.JPEG
  [AI] label=1  path=VQDM/train/ai/VQDM_1000_200_00_061_vqdm_00160.png
  [REAL] label=0  path=Midjourney/train/nature/n03379051_10304.JPEG
  [AI] label=1  path=Glide/train/ai/GLIDE_1000_200_07_778_glide_00140.png


### 5c · Fine-tune on GenImage subset

In [8]:
model, trainer_s1 = run_training_stage(
    model=model,
    processor=processor,
    train_ds=genimage_ds["train"],
    eval_ds=genimage_ds["validation"],
    output_dir=STAGE1_MODEL_DIR,
    epochs=EPOCHS_STAGE1,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    stage_name="Stage 1 – GenImage subset",
    fp16=USE_FP16,
)


  🚀  Stage 1 – GenImage subset


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.605341,0.498705,0.768000,0.767068,0.770161,0.764000
2,0.295572,0.389403,0.842000,0.837782,0.860759,0.816000
3,0.159763,0.396733,0.852000,0.850806,0.857724,0.844000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

***** train metrics *****
  epoch                    =         3.0
  total_flos               = 433020235GF
  train_loss               =      0.5088
  train_runtime            =  0:02:58.36
  train_samples_per_second =      33.639
  train_steps_per_second   =        1.06
  ✅  Model saved → outputs/models/genimage_stage1_finetuned


### 5d · Evaluate Stage 1 model on GenImage test split

In [9]:
s1_metrics = evaluate_model(
    model=model,
    processor=processor,
    test_ds=genimage_ds["test"],
    output_prefix="genimage_stage1",
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16,
)

  📄  Metrics saved → outputs/genimage_stage1_eval_results.json
  🖼️   Confusion matrix saved → outputs/genimage_stage1_confusion_matrix.png

  ────────────────────────────────────────
  Accuracy  : 0.8450
  Precision : 0.8513
  Recall    : 0.8360
  F1-score  : 0.8436
  ────────────────────────────────────────

              precision    recall  f1-score   support

        Real       0.84      0.85      0.85       500
AI-Generated       0.85      0.84      0.84       500

    accuracy                           0.84      1000
   macro avg       0.85      0.84      0.84      1000
weighted avg       0.85      0.84      0.84      1000



---
## 6 · Stage 2 – julienlucas Dataset

We **continue** fine-tuning the Stage 1 model (already updated in memory).

### 6a · Load the julienlucas dataset

In [ ]:
julienlucas_ds = load_julienlucas_dataset(
    val_ratio=0.1,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
julienlucas_ds

### 6b · Fine-tune (continued) on julienlucas

In [ ]:
model, trainer_s2 = run_training_stage(
    model=model,
    processor=processor,
    train_ds=julienlucas_ds["train"],
    eval_ds=julienlucas_ds["validation"],
    output_dir=STAGE2_MODEL_DIR,
    epochs=EPOCHS_STAGE2,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    stage_name="Stage 2 – julienlucas (with GenImage replay)",
    fp16=USE_FP16,
    replay_ds=genimage_ds["train"],
    replay_ratio=REPLAY_RATIO,
)

### 6c · Evaluate final model on julienlucas test split

In [ ]:
s2_metrics = evaluate_model(
    model=model,
    processor=processor,
    test_ds=julienlucas_ds["test"],
    output_prefix="julienlucas_stage2",
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16,
)

### 6d · (Optional) Re-evaluate final model on GenImage test

Check whether Stage 2 fine-tuning retained or degraded GenImage performance.

In [ ]:
s2_on_genimage = evaluate_model(
    model=model,
    processor=processor,
    test_ds=genimage_ds["test"],
    output_prefix="genimage_after_stage2",
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16,
)

---
## 7 · Save Weight Delta

Instead of committing the full ~327 MB model to Git, we save only the
**weight differences** (delta) from the base model. The resulting file
is typically ~82 MB (int8 quantised) and stays under GitHub's 100 MB limit.

> This is **full fine-tuning**, not LoRA/PEFT. The delta is not a
> lightweight adapter — it encodes changes across all parameters.
> If you switch to LoRA in the future the adapter files will be much
> smaller (~5–15 MB) and can be saved separately.

In [ ]:
# Reload module to pick up any fixes pushed after Colab cloned the repo
import importlib, visual_classifier
importlib.reload(visual_classifier)
from visual_classifier import save_weight_delta

delta_path, delta_mb = save_weight_delta(
    fine_tuned_model=model,
    base_model_name=BASE_MODEL,
    output_path="./fine_tuned_model_delta/weight_delta.pt",
)
print(f"\n✅ Delta saved to '{delta_path}' ({delta_mb:.2f} MB)")

---
## 8 · Summary

### What was saved

| Artefact | Path | Size |
|----------|------|------|
| Stage 1 full model | `outputs/models/genimage_stage1_finetuned/` | ~327 MB (gitignored) |
| Stage 2 full model | `outputs/models/genimage_then_julienlucas_stage2_finetuned/` | ~327 MB (gitignored) |
| Weight delta (int8) | `fine_tuned_model_delta/weight_delta.pt` | ~82 MB (tracked) |
| Stage 1 eval metrics | `outputs/genimage_stage1_eval_results.json` | tiny |
| Stage 2 eval metrics | `outputs/julienlucas_stage2_eval_results.json` | tiny |
| Post-S2 GenImage eval | `outputs/genimage_after_stage2_eval_results.json` | tiny |
| Confusion matrices | `outputs/*.png` | small |

### Next steps
- Run **`visual_classifier_eval.ipynb`** to compare original vs fine-tuned model.
- If running on Colab, download `outputs/` or push to GitHub.